In [ ]:
import logfire
from dotenv import load_dotenv
from pydantic_ai import Agent

load_dotenv()

# What the agent gets: a treasury it can top up, a card it can pay with, and its receipts.
from utils.rain_tools import AGENT_TOOLS, API_KEY, purchase_history, authorize_transaction,settle_transaction,list_transactions,create_payment_route, simulate_payment_route

assert API_KEY, "Put RAIN_API_KEY, RAIN_USER_ID and RAIN_CONTRACT_ID in agents/.env"

ImportError: cannot import name 'Agent' from 'pydantic' (/opt/anaconda3/lib/python3.13/site-packages/pydantic/__init__.py)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import logfire
logfire.configure()
logfire.instrument_pydantic_ai()

from pathlib import Path
from rain_agent import build_travel_agent, audit_purchases, DATA_PATH
import json

travel_agent = build_travel_agent()

In [ ]:
trip_request = Path("data01.json").read_text()
# In Jupyter you MUST await — do not use run_sync
res = await travel_agent.run(trip_request)
print(res.output)

In [ ]:
budget = json.loads(trip_request)["trip_request"]["budget"]["total_budget"]
audit_purchases(budget_usd=budget)

In [ ]:
res2 = await travel_agent.run(
    "Approved. Book and pay the plan you just proposed.",
    message_history=res.all_messages(),
)
print(res2.output)
audit_purchases(budget_usd=budget)

In [2]:
logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-eu.pydantic.dev/johanidler/rain

In [3]:
buyer = Agent(
    model='openai:gpt-5.6-luna',
    tools=AGENT_TOOLS,
    instructions="""You are the operations buyer for a small design studio and you hold
the company card. Orders you place are real and the money is spent for good.

Working a shopping list:
- Top the treasury up once, covering the whole list plus a little slack.
- Pay each merchant separately, with the merchant category code that fits it.
- If a payment is declined, fix the cause once and retry, then move on.
- Never spend more than the budget you were given.

Finish with a short summary: what you bought, what each cost, and the total.""",
)

In [ ]:
BUDGET_USD = 900

result = await buyer.run(f"""Restock the studio, budget ${BUDGET_USD}:

- 2 boxes of Chemex filters from Blue Bottle Coffee, $18.50 each
- 1 Herman Miller desk lamp from Design Within Reach, $245.00
- printer paper and pens from Staples, $86.40
- team lunch at Tartine Bakery, $112.75
- a spare Magic Trackpad from Apple Store, $129.00

Place the orders now.""")

print(result.output)

RuntimeError: This event loop is already running

In [ ]:
# How did it do? Read the spend back out of Rain, not out of the agent's answer.
# Step by step, tool call by tool call: the Logfire project URL printed above.
purchases = purchase_history(limit=20)

for purchase in purchases:
    print(f"{purchase['amount_usd']:>8.2f}  {purchase['status']:<10} {purchase['merchant']}")

total = sum(p["amount_usd"] for p in purchases)
print(f"\n{total:>8.2f}  total of {len(purchases)} payments — budget ${BUDGET_USD}")
print("over budget!" if total > BUDGET_USD else "within budget")